# Teste em etapas: instalar → usar get(url) → desinstalar

Execute as células **na ordem** para:
1. Instalar o pacote localmente
2. Chamar a função `get` com uma URL da API SIDRA (IBGE)
3. Desinstalar o pacote

**Requisito:** abrir/executar o notebook com o diretório de trabalho na **raiz do projeto** (`c:\BAU\data_economist`).

In [ ]:
import os
import sys
if os.path.basename(os.getcwd()) == "tests":
    os.chdir("..")
raiz = os.path.abspath(".")
src = os.path.join(raiz, "src")
if os.path.isdir(src) and src not in sys.path:
    sys.path.insert(0, src)
for key in list(sys.modules.keys()):
    if key == "data_economist" or key.startswith("data_economist."):
        del sys.modules[key]
print("Diretório:", os.getcwd())

## Etapa 1: Instalar localmente

Instala o `data_economist` em modo editável no **mesmo Python que o kernel** do notebook usa, para o `import` na Etapa 2 funcionar.

In [ ]:
# Usar o mesmo Python do kernel para não instalar noutro interpretador
import sys
!{sys.executable} -m pip install -e .

## Etapa 2: Rodar ibge.url(url)

Importa o módulo `ibge` e chama `ibge.url(url)`. Pode passar uma URL ou uma lista de URLs `[url1, url2]`; com lista, o resultado é um JSON aninhado (lista de resultados na mesma ordem).

*Se der `ModuleNotFoundError`, a célula coloca a pasta `src/` do projeto em `sys.path` para o kernel encontrar o código.*

In [ ]:
# Garantir que o projeto está no path e forçar recarga do módulo (evita cache antigo sem ibge.url)
import sys
import os
raiz = os.path.abspath(".." if os.path.basename(os.getcwd()) == "tests" else ".")
src = os.path.join(raiz, "src")
if os.path.isdir(src) and src not in sys.path:
    sys.path.insert(0, src)
# Sempre limpar cache para o import usar o código atual (com ibge.url)
for key in list(sys.modules.keys()):
    if key == "data_economist" or key.startswith("data_economist."):
        del sys.modules[key]

from data_economist import ibge

# Uma URL ou [url, url] — resultado sempre JSON (com lista: aninhado)
url = (
    "https://apisidra.ibge.gov.br/values/t/8888/n3/all/v/12606/p/first/c544/129317/d/v12606%205",
    "https://apisidra.ibge.gov.br/values/t/8888/n3/all/v/12606/p/last/c544/129317/d/v12606%205"
)

dados = ibge.url(url)
print(f"Recebidos {len(dados)} registos (1º = cabeçalho).")
print("Primeiros 2 elementos:")
dados[:2]

In [ ]:
# DataFrame com TODOS os dados: uma URL = [cabeçalho, reg...]; várias URLs = [resultado1, resultado2, ...]
# Colunas SIDRA: NC, NN, MC, MN, V e D1C, D1N, ..., D9C, D9N
import pandas as pd
todos_registos = []
if isinstance(dados[0], list):
    # Várias URLs: cada elemento de dados é um resultado (cabeçalho + registos)
    for resultado in dados:
        todos_registos.extend(resultado[1:])
else:
    # Uma URL: dados = [cabeçalho, reg1, reg2, ...]
    todos_registos = dados[1:]
df = pd.DataFrame(todos_registos)
print(f"DataFrame: {len(df)} linhas × {len(df.columns)} colunas")
df

## Etapa 3: Desinstalar

Remove o pacote do ambiente com `pip uninstall`.

**Por que `ibge.url()` ainda funciona depois?**  
O Jupyter mantém os módulos **em memória** enquanto o kernel está ligado. O `pip uninstall` só remove o pacote do disco; não descarrega o que já foi importado. Por isso, mesmo após desinstalar, `ibge.url()` continua a funcionar nesta sessão.

**Para ver que o pacote foi mesmo desinstalado:** reinicie o kernel (menu *Kernel → Restart* ou *Reiniciar*). Depois, se tentar `from data_economist import ibge` sem instalar de novo, dará `ModuleNotFoundError`.

In [ ]:
!pip uninstall data-economist -y